<img src="logo.png" alt="Vegeta" width="240">

# Unmanned ground vehicle — a small rover for rugged terrain, mechanically

A 3 kg four-wheel rover with trailing-arm suspension, printed in PA12-CF, for rocky trails. The
notebook covers the mechanics end to end:

```
CAD (Dedalus): chassis tub, four trailing arms, wheels ─► masses, wheel loads at rest
terrain profiles (ISO 8608 classes + rocks + a drop) ─► quarter-car dynamics (numpy) ─► wheel force histories
peak loads ─► static FEA of the arm and the chassis in torsion (Talos), arm modes with the wheel mass
force histories ─► rainflow (Chronos) ─► spectra for three terrains ─► fatigue on the arm's stress field
fleet usage ─► life ─► print the arm (Mellonia)
```

Every input is explicit and coarse (spring rates, tyre stiffness, an assumed S-N curve). Compare,
record, then test on a real trail.

In [ ]:
import json, math, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from vegeta import dedalus, talos, chronos, mellonia
from vegeta.dedalus import viz as dviz
from vegeta.talos import viz as tviz
from vegeta.mellonia import viz as mviz
from vegeta.mellonia.examples import GENERIC_PLA_0_2MM

RUNS = Path("_runs/rover"); shutil.rmtree(RUNS, ignore_errors=True); RUNS.mkdir(parents=True)
design_file = RUNS / "rover.py"
shutil.copy(Path("designs/rover.py"), design_file)
rover_design = dedalus.load_design(f"{design_file}:Rover")
pd.DataFrame(rover_design.params.table()).set_index("name")

## 1. The vehicle

In [ ]:
p = rover_design.resolve()
rover = rover_design.generate()
arm = rover_design.generate(part="arm")
chassis = rover_design.generate(part="chassis")
cad = {k: g.export(RUNS / "cad" / k, stl_tolerance=0.05) for k, g in (("rover", rover), ("arm", arm), ("chassis", chassis))}
dviz.show(dviz.plot3d(rover))

In [ ]:
fig = dviz.plot_sections(rover, normal="y", positions=[0.0, p["width"] / 2 + 8, p["width"] / 2 + 45], cols=3)   # centre, arm plane, wheel plane
fig = dviz.plot_sections(arm, normal="x", positions=[10.0, 45.0, 85.0], cols=3)

In [ ]:
RHO_PA12CF = 1.1e-3       # g/mm^3
parts = pd.DataFrame([
    ("chassis (PA12-CF, from CAD)", chassis.volume * RHO_PA12CF),
    ("suspension arms 4x (from CAD)", 4 * arm.volume * RHO_PA12CF),
    ("wheels + tyres 4x", 4 * 120.0),
    ("motors + gearboxes 4x", 4 * 95.0),
    ("battery 3S 5000 mAh", 380.0),
    ("controller, radio, camera", 180.0),
    ("wiring, bolts, bearings", 120.0),
], columns=["part", "mass_g"]).set_index("part")
M_TOTAL = parts["mass_g"].sum() / 1000
M_WHEEL = (120.0 + 95.0 * 0.5) / 1000                  # unsprung mass per corner: wheel + half the motor/gearbox
M_SPRUNG_CORNER = (M_TOTAL - 4 * M_WHEEL) / 4
G = 9.81
print(f"total {M_TOTAL:.2f} kg | sprung per corner {M_SPRUNG_CORNER:.3f} kg | static wheel load {M_TOTAL * G / 4:.1f} N")
parts.round(1)

## 2. Terrain and wheel loads: a quarter-car model over three terrains

Each corner is a two-mass system: the sprung quarter of the body on the suspension (a torsion spring
at the arm pivot, expressed as a vertical rate at the wheel, with a damper) and the wheel on its tyre
stiffness, driven by the ground profile under the wheel. Profiles are ISO 8608 classes generated
from their PSD (seeded), with rocks (half-sine bumps) and a drop added for the rough missions.
Tyre force below zero means the wheel left the ground.

In [ ]:
K_SUSP = 520.0       # N/m at the wheel (torsion spring at the pivot), sag ~ 14 mm under the static load
ZETA = 0.3           # damping ratio of the suspension (damper or friction at the pivot)
K_TYRE = 6000.0      # N/m, foam-filled tyre
C_SUSP = 2 * ZETA * math.sqrt(K_SUSP * M_SPRUNG_CORNER)

def iso8608_profile(length_m, dx, gd_n0, seed, n0=0.1, n_min=0.02, n_max=8.0, n_waves=400):
    # displacement PSD Gd(n) = Gd(n0) (n/n0)^-2, superposition of sinusoids with random phase
    rng = np.random.default_rng(seed)
    x = np.arange(0, length_m, dx)
    ns = np.linspace(n_min, n_max, n_waves)
    dn = ns[1] - ns[0]
    amps = np.sqrt(2 * gd_n0 * (ns / n0) ** -2 * dn)
    phases = rng.uniform(0, 2 * math.pi, n_waves)
    z = (amps[None, :] * np.sin(2 * math.pi * ns[None, :] * x[:, None] + phases[None, :])).sum(axis=1)
    return x, z

def add_rocks(x, z, height_m, width_m, spacing_m, seed):
    rng = np.random.default_rng(seed)
    z = z.copy()
    centres = np.arange(spacing_m, x[-1] - spacing_m, spacing_m)
    for xc in centres + rng.uniform(-0.3, 0.3, len(centres)):
        mask = np.abs(x - xc) < width_m / 2
        z[mask] += height_m * np.cos(math.pi * (x[mask] - xc) / width_m)
    return z

def add_drop(x, z, at_m, depth_m):
    z = z.copy(); z[x > at_m] -= depth_m
    return z

def quarter_car(x, z_road, speed, dt=5e-4):
    # states: sprung z_s, wheel z_w (relative to static equilibrium), velocities; ground z_r(t) = z_road(x = v t)
    t = np.arange(0, x[-1] / speed, dt)
    zr = np.interp(t * speed, x, z_road)
    zs = zw = vs = vw = 0.0
    Fs, Ft, As, travel = np.zeros_like(t), np.zeros_like(t), np.zeros_like(t), np.zeros_like(t)
    for i, z_r in enumerate(zr):
        f_susp = K_SUSP * (zw - zs) + C_SUSP * (vw - vs)                     # suspension force on the body (+ up)
        f_tyre = max(K_TYRE * (z_r - zw), -M_TOTAL * G / 4)                  # tyre can only push (down to zero contact)
        a_s = f_susp / M_SPRUNG_CORNER
        a_w = (f_tyre - f_susp) / M_WHEEL
        vs += a_s * dt; vw += a_w * dt; zs += vs * dt; zw += vw * dt
        Fs[i], Ft[i], As[i], travel[i] = f_susp + M_SPRUNG_CORNER * G, f_tyre + M_TOTAL * G / 4, a_s, zw - zs
    return t, zr, Fs, Ft, As, travel

terrains = {
    "paved patrol":  {"class": "A", "gd": 16e-6,   "speed": 1.5, "length": 300.0, "rocks": None,                "drop": None,        "seed": 1},
    "gravel trail":  {"class": "C", "gd": 256e-6,  "speed": 1.2, "length": 300.0, "rocks": (0.02, 0.10, 6.0),   "drop": None,        "seed": 2},
    "rocky field":   {"class": "E", "gd": 4096e-6, "speed": 0.8, "length": 200.0, "rocks": (0.04, 0.12, 2.5),   "drop": (150.0, 0.12), "seed": 3},
}
sims = {}
for name, tr in tqdm(terrains.items(), desc="terrains"):
    x, z = iso8608_profile(tr["length"], 0.005, tr["gd"], tr["seed"])
    if tr["rocks"]:
        z = add_rocks(x, z, *tr["rocks"], seed=tr["seed"] + 10)
    if tr["drop"]:
        z = add_drop(x, z, *tr["drop"])
    t, zr, Fs, Ft, As, travel = quarter_car(x, z, tr["speed"])
    sims[name] = dict(t=t, zr=zr, Fs=Fs, Ft=Ft, As=As, travel=travel, duration_s=t[-1], speed=tr["speed"], length=tr["length"])
summary = pd.DataFrame({k: {"speed_m_s": s["speed"], "duration_s": s["duration_s"], "tyre_force_max_N": s["Ft"].max(),
                            "tyre_force_min_N": s["Ft"].min(), "arm_force_max_N": s["Fs"].max(),
                            "body_accel_rms_g": np.sqrt(np.mean(s["As"] ** 2)) / G, "travel_max_mm": np.abs(s["travel"]).max() * 1000,
                            "airborne_%": 100 * np.mean(s["Ft"] <= 1e-6)} for k, s in sims.items()}).T.round(2)
summary

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(13, 9))
for (name, s), (a1, a2) in zip(sims.items(), axes):
    a1.plot(s["t"] * s["speed"], s["zr"] * 1000, lw=0.6, color="#5a6a7a"); a1.set(title=f"{name}: ground profile", xlabel="distance [m]", ylabel="height [mm]")
    a2.plot(s["t"], s["Ft"], lw=0.5, label="tyre force"); a2.plot(s["t"], s["Fs"], lw=0.5, label="arm (suspension) force")
    a2.set(title=f"{name}: wheel forces", xlabel="time [s]", ylabel="N"); a2.legend(fontsize=8)
    for a in (a1, a2): a.grid(alpha=0.3)
fig.tight_layout()

The chassis feels the arm force; the arm feels the tyre force at its axle and, at every rock edge,
a longitudinal component ≈ vertical force × local slope. That longitudinal history is derived from the
profile slope where the tyre is loaded.

In [ ]:
for name, s in sims.items():
    slope = np.gradient(s["zr"], s["t"] * s["speed"])
    s["Fx"] = np.clip(s["Ft"] * np.clip(np.abs(slope), 0, 1.5), 0, None)     # rock strikes: longitudinal load on the arm
print({k: round(float(s["Fx"].max()), 1) for k, s in sims.items()}, "N peak longitudinal")

## 3. Static strength at the peak loads (Talos)

Suspension arm: pivot bore fixed, tyre peak load at the axle bore, vertical and longitudinal
(the rocky-field drop). Chassis: torsion — two diagonal pivots fixed, the other two loaded by the arm
force (one wheel on a rock, the opposite one in a hole).

In [ ]:
PA12CF = talos.Material("PA12-CF (printed)", youngs_modulus=3500.0, poissons_ratio=0.40, density=1.1e-9, yield_strength=60.0,
                        source="nominal datasheet, flat orientation; assume 30 % less across layers")
L, h = p["arm_length"], p["arm_height"]
rp, ra = p["pivot_diameter"] / 2 + 0.2, p["axle_diameter"] / 2 + 0.2
ARM_REGIONS = [talos.SurfacesInBox("pivot", (-rp, -h, -rp, rp, h, rp)), talos.SurfacesInBox("axle", (L - ra, -h, -ra, L + ra, h, ra))]
WHEEL_MASS_T = M_WHEEL * 1e-3                                                  # tonnes at the axle

def arm_model(loads, name, masses=()):
    return talos.StructuralModel(cad["arm"].artifacts["step"], "mm-N-MPa", PA12CF, ARM_REGIONS, [talos.FixedSupport("pivot")], loads,
                                 talos.MeshSettings(element_size=2.0), name=name, masses=list(masses))

peak_z = max(s["Ft"].max() for s in sims.values())
peak_x = max(s["Fx"].max() for s in sims.values())
arm_peak = arm_model([talos.Force("axle", fz=peak_z, fx=peak_x)], "arm_peak")
mesh_arm = arm_peak.mesh(RUNS / "arm_peak", progress=True)
res_arm = arm_peak.solve(RUNS / "arm_peak")
print(res_arm)
tviz.show(tviz.plot_problem(arm_peak, mesh_arm))

In [ ]:
if res_arm.ok:
    tviz.show(tviz.plot_results(res_arm, field="von_mises"))

In [ ]:
px, pz = p["length"] / 2 - p["pivot_x"], -p["plate_thickness"] - p["rail_height"] / 2
rr = p["pivot_diameter"] / 2 + 0.3
CH_REGIONS = [talos.SurfacesInBox(f"pivot_{'f' if sx > 0 else 'r'}{'l' if sy > 0 else 'r'}",
                                  (sx * px - rr, sy * p["width"] / 2 - 10, pz - rr, sx * px + rr, sy * p["width"] / 2 + 10, pz + rr))
              for sx in (-1, 1) for sy in (-1, 1)]
arm_peak_force = max(s["Fs"].max() for s in sims.values())
chassis_model = talos.StructuralModel(cad["chassis"].artifacts["step"], "mm-N-MPa", PA12CF, CH_REGIONS,
                                      [talos.FixedSupport("pivot_rl"), talos.FixedSupport("pivot_fr")],
                                      [talos.Force("pivot_fl", fz=-arm_peak_force), talos.Force("pivot_rr", fz=-arm_peak_force)],
                                      talos.MeshSettings(element_size=6.0), name="chassis_torsion")
chassis_model.mesh(RUNS / "chassis_torsion", progress=True)
res_ch = chassis_model.solve(RUNS / "chassis_torsion")
print(res_ch)
if res_ch.ok:
    tviz.show(tviz.plot_results(res_ch, field="von_mises"))

In [ ]:
pd.DataFrame({"arm at peak wheel load": {"load_N": f"{peak_z:.0f} up, {peak_x:.0f} longitudinal", "max_von_mises_MPa": res_arm.metrics.get("max_von_mises"),
                                         "SF_yield": res_arm.metrics.get("safety_factor_yield"), "deflection_mm": res_arm.metrics.get("max_displacement")},
              "chassis torsion": {"load_N": f"{arm_peak_force:.0f} on two diagonal pivots", "max_von_mises_MPa": res_ch.metrics.get("max_von_mises"),
                                  "SF_yield": res_ch.metrics.get("safety_factor_yield"), "deflection_mm": res_ch.metrics.get("max_displacement")}}).T

## 4. Arm modes with the wheel on it, against the terrain excitation band

Terrain excites the wheel at frequencies up to `speed / shortest wavelength` (≈ 1.5 m/s / 0.125 m
= 12 Hz for the ISO band used) plus the wheel-hop mode of the quarter car; the arm's own bending mode
must sit well above both, or the arm rings on every rock.

In [ ]:
arm_modal = arm_model([], "arm_modal", masses=[talos.PointMass("axle", WHEEL_MASS_T)])
arm_modal.mesh(RUNS / "arm_modal")
modes = arm_modal.solve_modes(RUNS / "arm_modal", n_modes=4)
f_hop = math.sqrt((K_TYRE + K_SUSP) / M_WHEEL) / (2 * math.pi)
f_body = math.sqrt(K_SUSP / M_SPRUNG_CORNER) / (2 * math.pi)
print(f"arm modes with the wheel: {[round(f, 1) for f in modes.metrics['frequencies_hz']]} Hz | body bounce {f_body:.1f} Hz | wheel hop {f_hop:.1f} Hz | terrain band up to {1.5 / 0.125:.0f} Hz")
structure = chronos.Structure(tuple(modes.metrics["frequencies_hz"]), damping_ratio=0.04)
tviz.show(tviz.plot_mode(modes, mode=1))

## 5. Cyclic loads: rainflow on the wheel-force histories (Chronos), one spectrum per terrain

The simulated histories are counted directly (ASTM rainflow) into blocks of the `wheel_z` pattern
(tyre force on the axle) and `wheel_x` (rock strikes). The time simulated is a few minutes; the blocks
are scaled to a mission of the stated duration.

In [ ]:
MISSION_MIN = {"paved patrol": 40.0, "gravel trail": 30.0, "rocky field": 20.0}
spectra = {}
for name, s in sims.items():
    scale = MISSION_MIN[name] * 60 / s["duration_s"]
    blocks = []
    for pattern, series in (("wheel_z", s["Ft"]), ("wheel_x", s["Fx"])):
        merged = {}
        for rng_, mean, count in chronos.rainflow(series[::4]):                  # 2 kHz -> 500 Hz, ample for < 30 Hz content
            if rng_ < 0.5:                                                        # ignore sub-0.5 N noise
                continue
            key = (round(mean, 1), round(rng_ / 2, 1))
            merged[key] = merged.get(key, 0.0) + count * scale
        blocks += [chronos.Block(pattern, m, a, c, f"{name}: {pattern} rainflow") for (m, a), c in merged.items()]
    spectra[name] = chronos.LoadSpectrum(name, MISSION_MIN[name] * 60, blocks, ["wheel_z", "wheel_x"],
                                         notes=f"quarter-car over {s['length']:.0f} m at {s['speed']} m/s, scaled to {MISSION_MIN[name]:.0f} min")
    spectra[name].save(RUNS / f"spectrum_{name.replace(' ', '_')}.json")
    fig = spectra[name].plot()
pd.DataFrame({k: {"blocks": len(sp.blocks), "cycles_per_mission": sp.total_cycles, "max_amplitude_N": max(b.amplitude for b in sp.blocks)} for k, sp in spectra.items()}).T.round(1)

## 6. Fatigue of the arm, damage map, life over a fleet usage

Unit cases: 100 N vertical and 100 N longitudinal at the axle. S-N for printed PA12-CF — **assumed**:
σ_f = 110 MPa, b = −0.10, Goodman with 90 MPa ultimate (flat orientation; across layers use less).

In [ ]:
unit_models = {"wheel_z": (arm_model([talos.Force("axle", fz=100.0)], "unit_z"), 100.0),
               "wheel_x": (arm_model([talos.Force("axle", fx=100.0)], "unit_x"), 100.0)}
unit_cases = {}
for k, (m, load) in unit_models.items():
    m.mesh(RUNS / k); unit_cases[k] = (m.solve(RUNS / k), load)
    print(k, unit_cases[k][0].status, round(unit_cases[k][0].metrics["max_von_mises"], 2), "MPa per 100 N")
CURVE = talos.FatigueCurve("PA12-CF (assumed)", sigma_f=110.0, b=-0.10, ultimate=90.0, source="assumed; coupon tests needed")
fatigue = {k: talos.assess_fatigue(unit_cases, sp.to_dict(), CURVE, workdir=RUNS / f"fatigue_{k.replace(' ', '_')}") for k, sp in spectra.items()}
life = pd.DataFrame({k: {"mission_min": MISSION_MIN[k], "damage_per_mission": f.result.metrics["damage_per_pass"],
                         "missions_to_failure": f.result.metrics["passes_to_failure"], "hours_to_failure": f.result.metrics["hours_to_failure"]}
                     for k, f in fatigue.items()}).T
life

In [ ]:
worst = life["damage_per_mission"].astype(float).idxmax()
tviz.show(tviz.plot_damage(fatigue[worst], unit_cases["wheel_z"][0].artifacts["mesh"]))

In [ ]:
damage = {k: f.result.metrics["damage_per_pass"] for k, f in fatigue.items()}
hours = {k: MISSION_MIN[k] / 60 for k in sims}
usage = {"paved patrol": 0.4, "gravel trail": 0.4, "rocky field": 0.2}
rate = sum(usage[k] * damage[k] for k in usage) / sum(usage[k] * hours[k] for k in usage)
sim = chronos.simulate_life(damage, hours, usage, n_flights=5000, seed=0)
fig = sim.plot()
print(f"usage {usage}: damage per 1000 h {rate * 1000:.3g} -> " + (f"{1 / rate:.0f} h to failure" if rate > 1e-9 else "not life-limiting (> 1e9 h)"))
mixes = {"as above": usage, "rocky-only": {"paved patrol": 0.0, "gravel trail": 0.0, "rocky field": 1.0}, "paved-only": {"paved patrol": 1.0, "gravel trail": 0.0, "rocky field": 0.0}}
pd.DataFrame({n: {"damage per 1000 h": sum(m[k] * damage[k] for k in m) / sum(m[k] * hours[k] for k in m) * 1000} for n, m in mixes.items()}).T

## 7. Print the arm (Mellonia)

Flat on its side (bores vertical): the layers then run along the arm, in the direction of the
bending stress, which is what the flat-orientation material values assume.

In [ ]:
prn = mellonia.slice_stl(cad["arm"].artifacts["stl"], GENERIC_PLA_0_2MM, mellonia.Orientation(rotate_x=90), RUNS / "print_arm")
print(prn)
if prn.ok:
    mviz.show(mviz.plot_toolpath(prn))

## 8. Export

In [ ]:
summary_doc = {"design": {"file": "designs/rover.py", "parameters": p}, "mass_kg": M_TOTAL,
               "suspension": {"k_susp_N_m": K_SUSP, "zeta": ZETA, "k_tyre_N_m": K_TYRE, "m_wheel_kg": M_WHEEL},
               "terrains": {k: {kk: v for kk, v in tr.items()} for k, tr in terrains.items()},
               "dynamics": summary.to_dict(), "arm_modes_hz": modes.metrics["frequencies_hz"],
               "static": {"arm_peak": res_arm.metrics, "chassis_torsion": res_ch.metrics},
               "curve": CURVE.__dict__, "damage_per_mission": damage, "hours_per_mission": hours, "usage": usage,
               "damage_per_1000h": rate * 1000, "spectra": {k: str(RUNS / f"spectrum_{k.replace(' ', '_')}.json") for k in spectra}}
(RUNS / "rover_mechanics.json").write_text(json.dumps(summary_doc, indent=2, default=float))
print("written:", sorted(x.name for x in RUNS.iterdir()))

**Next steps an engineer would take:** measure the real spring rate and tyre stiffness (they set the
peak loads more than anything in the CAD); add the motor torque reaction and cornering loads to the
arm; run the chassis drop case with the battery mass; and, if the rocky-only column is the limiting
one, thicken the arm root or add a fillet at the pivot boss (the hotspot in the damage map) — a
revision in a `vegeta.core` workspace, or a campaign with `damage_per_1000h` as the objective.